# Quest 4

In [101]:
import pandas as pd
import numpy as np
import requests

## 1) Read the JSON file that you saved in ex02

In [102]:
src_file = "../ex02/auto.json"
pd.options.display.float_format = "{:,.2f}".format
df = pd.read_json(src_file)
display(df)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,"3,200.00",Ford,Focus
1,E432XX77RUS,1,"6,500.00",Toyota,Camry
2,7184TT36RUS,1,"2,100.00",Ford,Focus
3,X582HE161RUS,2,"2,000.00",Ford,Focus
5,92918M178RUS,1,"5,700.00",Ford,Focus
...,...,...,...,...,...
926,Y163O8161RUS,2,"1,600.00",Ford,Focus
927,M0309X197RUS,1,"22,300.00",Ford,Focus
928,O673E8197RUS,2,600.00,Ford,Focus
929,8610T8154RUS,1,"2,000.00",Ford,Focus


## 2) Enrich the dataframe using a sample from that dataframe

* create a sample with 200 new observations with random_state = 21

In [103]:
sample_df = df.sample(n=200, random_state=21)
display(sample_df)

,CarNumber,Refund,Fines,Make,Model
629,M0299X197RUS,2,"19,200.00",Ford,Focus
32,83298C154RUS,2,"8,594.59",Ford,Focus
140,H957HY161RUS,1,"2,000.00",Ford,Focus
261,T941CC96RUS,1,"2,000.00",Ford,Focus
903,H966HY161RUS,1,500.00,Ford,Focus
...,...,...,...,...,...
19,8182XX154RUS,1,200.00,Ford,Focus
828,X796TH96RUS,1,500.00,Ford,Focus
692,T011MY163RUS,2,"4,000.00",Ford,Focus
735,T341CC96RUS,2,"1,000.00",Volkswagen,Passat


* there are no restrictions on the refund and fines, you can take any value from these columns at random and use it towards any car number:

In [104]:
un_Refund = df["Refund"].unique()
un_Fines = df["Fines"].unique()

sample_df["Refund"] = np.random.choice(a=un_Refund, size=len(sample_df))
sample_df["Fines"] = np.random.choice(a=un_Fines, size=len(sample_df))

display(sample_df)

,CarNumber,Refund,Fines,Make,Model
629,M0299X197RUS,2,"7,400.00",Ford,Focus
32,83298C154RUS,2,"10,100.00",Ford,Focus
140,H957HY161RUS,1,"9,900.00",Ford,Focus
261,T941CC96RUS,1,"7,200.00",Ford,Focus
903,H966HY161RUS,1,"25,900.00",Ford,Focus
...,...,...,...,...,...
19,8182XX154RUS,2,"75,900.00",Ford,Focus
828,X796TH96RUS,1,"3,100.00",Ford,Focus
692,T011MY163RUS,1,"69,600.00",Ford,Focus
735,T341CC96RUS,1,"27,800.00",Volkswagen,Passat


* concatenate the sample with the initial dataframe to a new dataframe:

In [105]:
concat_rows = pd.concat(objs=[df,sample_df], join="inner", ignore_index=True)
display(concat_rows)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,"3,200.00",Ford,Focus
1,E432XX77RUS,1,"6,500.00",Toyota,Camry
2,7184TT36RUS,1,"2,100.00",Ford,Focus
3,X582HE161RUS,2,"2,000.00",Ford,Focus
4,92918M178RUS,1,"5,700.00",Ford,Focus
...,...,...,...,...,...
920,8182XX154RUS,2,"75,900.00",Ford,Focus
921,X796TH96RUS,1,"3,100.00",Ford,Focus
922,T011MY163RUS,1,"69,600.00",Ford,Focus
923,T341CC96RUS,1,"27,800.00",Volkswagen,Passat


## 3) Enrich the dataframe concat_rows by a new column with the data generated:

* create a series with the name Year using random integers from 1980 to 2019
* use np.random.seed(21) before generating the years

In [106]:
np.random.seed(21)
Year = pd.Series(np.random.randint(low=1980, high=2019, size=len(concat_rows)), name="Year")
print(Year)

0      1989
1      1995
2      1984
3      2015
4      2014
       ... 
920    1996
921    2002
922    1996
923    2012
924    1984
Name: Year, Length: 925, dtype: int64


* concatenate the series with the dataframe and name it __fines__:

In [107]:
fines = pd.concat([concat_rows, Year], axis='columns')
fines

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995
2,7184TT36RUS,1,"2,100.00",Ford,Focus,1984
3,X582HE161RUS,2,"2,000.00",Ford,Focus,2015
4,92918M178RUS,1,"5,700.00",Ford,Focus,2014
...,...,...,...,...,...,...
920,8182XX154RUS,2,"75,900.00",Ford,Focus,1996
921,X796TH96RUS,1,"3,100.00",Ford,Focus,2002
922,T011MY163RUS,1,"69,600.00",Ford,Focus,1996
923,T341CC96RUS,1,"27,800.00",Volkswagen,Passat,2012


## 4) Enrich the dataframe with the data from another dataframe

### 1. Create a new dataframe with the car numbers and their owners

Select all surnames given in surname.json:

In [108]:
src_file_q4 = "../../datasets/surname.json"
un_surnames = pd.read_json(src_file_q4, orient='values')
un_surnames.columns = un_surnames.iloc[0]   # устоновим 0 строчку как заголовок
un_surnames.drop(0, inplace=True)
display(un_surnames)

,NAME,COUNT,RANK
1,ADAMS,427865,42
2,ALLEN,482607,33
3,ALVAREZ,233983,92
4,ANDERSON,784404,15
5,BAILEY,277845,72
...,...,...,...
96,WILLIAMS,1625252,3
97,WILSON,801882,14
98,WOOD,250715,84
99,WRIGHT,458980,35


Чтобы создать dataframe {"CarNumber", "Name"}, отберем сначала уникальные номера машин из предыдущей таблицы:

In [109]:
un_car_numbers = concat_rows.drop_duplicates("CarNumber")["CarNumber"].reset_index(drop=True)
display(un_car_numbers)

0      Y163O8161RUS
1       E432XX77RUS
2       7184TT36RUS
3      X582HE161RUS
4      92918M178RUS
           ...     
526    O136HO197RUS
527    O22097197RUS
528    M0309X197RUS
529    O673E8197RUS
530    8610T8154RUS
Name: CarNumber, Length: 531, dtype: object

Создадим столько же рандомных имен:

In [110]:
random_surnames = un_surnames["NAME"].sample(n=len(un_car_numbers), random_state=21, replace=True).reset_index(drop=True)
display(random_surnames)

0      RICHARDSON
1            ROSS
2          MORGAN
3          BAILEY
4           LOPEZ
          ...    
526      CAMPBELL
527          HALL
528         BAKER
529          DIAZ
530        MORGAN
Name: NAME, Length: 531, dtype: object

In [111]:
owners = pd.concat(objs=[un_car_numbers, random_surnames], axis="columns")
owners.columns = ["CarNumber", "SURNAME"]
display(owners)

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
526,O136HO197RUS,CAMPBELL
527,O22097197RUS,HALL
528,M0309X197RUS,BAKER
529,O673E8197RUS,DIAZ


## 5) Append 5 more observations to the fines dataframe (come up with your own ideas of CarNumber, etc.)

In [112]:
display(fines)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995
2,7184TT36RUS,1,"2,100.00",Ford,Focus,1984
3,X582HE161RUS,2,"2,000.00",Ford,Focus,2015
4,92918M178RUS,1,"5,700.00",Ford,Focus,2014
...,...,...,...,...,...,...
920,8182XX154RUS,2,"75,900.00",Ford,Focus,1996
921,X796TH96RUS,1,"3,100.00",Ford,Focus,2002
922,T011MY163RUS,1,"69,600.00",Ford,Focus,1996
923,T341CC96RUS,1,"27,800.00",Volkswagen,Passat,2012


In [113]:
new_cars = [["I837GX83923RUS", 1, 21000, "Toyota", "Focus", 1999],
            ["X777XX7777RUS", 2, 3000, "Volkswagen", "Focus", 2000],
            ["T834DJ9384734RUS",1, 1500, "Toyota", "Camry", 1998],
            ["X000Q0000RUS", 2, 40000, "Ford", "Passat", 2004],
            ["89384CC374RUS",1, 1000, "Toyota", "Camry", 2025]]
new_fines = pd.DataFrame(new_cars, columns=fines.columns)

#fines = pd.concat(objs=[fines, new_fines], ignore_index=True)      # через concat
fines = fines.append(new_fines, ignore_index=True)
display(fines)


,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995
2,7184TT36RUS,1,"2,100.00",Ford,Focus,1984
3,X582HE161RUS,2,"2,000.00",Ford,Focus,2015
4,92918M178RUS,1,"5,700.00",Ford,Focus,2014
...,...,...,...,...,...,...
925,I837GX83923RUS,1,"21,000.00",Toyota,Focus,1999
926,X777XX7777RUS,2,"3,000.00",Volkswagen,Focus,2000
927,T834DJ9384734RUS,1,"1,500.00",Toyota,Camry,1998
928,X000Q0000RUS,2,"40,000.00",Ford,Passat,2004


## 6) Delete the dataframe last 20 observations from the owners and add 3 new observations (they are not the same as those you add to the fines dataframe)

In [114]:
print(f"Количество строк тогда: {len(owners)}")
owners.drop(owners.tail(20).index, inplace=True)
print(f"Количество строк сейчас: {len(owners)}")
display(owners)

Количество строк тогда: 531
Количество строк сейчас: 511


,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
506,T914CT197RUS,HERNANDEZ
507,E41977152RUS,BAKER
508,9464EX178RUS,MARTIN
509,O50197197RUS,WRIGHT


In [115]:
new_owners = [["O938284829RUS", "ANNA"],
              ["P394XXX93hRUS", "SAVVA"],
              ["T93848Dn38RUS", "MARTIN"]]
new_owners = pd.DataFrame(new_owners, columns=owners.columns)
owners = owners.append(new_owners, ignore_index=True)
print(f"Количество строк сейчас: {len(owners)}")
display(owners)

Количество строк сейчас: 514


,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
509,O50197197RUS,WRIGHT
510,7608EE777RUS,HILL
511,O938284829RUS,ANNA
512,P394XXX93hRUS,SAVVA


## 7) Join both dataframes

In [116]:
display(fines)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995
2,7184TT36RUS,1,"2,100.00",Ford,Focus,1984
3,X582HE161RUS,2,"2,000.00",Ford,Focus,2015
4,92918M178RUS,1,"5,700.00",Ford,Focus,2014
...,...,...,...,...,...,...
925,I837GX83923RUS,1,"21,000.00",Toyota,Focus,1999
926,X777XX7777RUS,2,"3,000.00",Volkswagen,Focus,2000
927,T834DJ9384734RUS,1,"1,500.00",Toyota,Camry,1998
928,X000Q0000RUS,2,"40,000.00",Ford,Passat,2004


In [117]:
display(owners)

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
509,O50197197RUS,WRIGHT
510,7608EE777RUS,HILL
511,O938284829RUS,ANNA
512,P394XXX93hRUS,SAVVA


* the new dataframe should have only the car numbers that exist in both dataframes - _Inner Join_

In [118]:
df_join1 = pd.merge(left=fines, right=owners, on="CarNumber", how="inner")
display(df_join1)

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989,RICHARDSON
1,Y163O8161RUS,2,"1,600.00",Ford,Focus,1999,RICHARDSON
2,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995,ROSS
3,E432XX77RUS,2,"13,000.00",Toyota,Camry,1992,ROSS
4,7184TT36RUS,1,"2,100.00",Ford,Focus,1984,MORGAN
...,...,...,...,...,...,...,...
894,E41977152RUS,2,"2,400.00",Ford,Focus,2001,BAKER
895,9464EX178RUS,2,"2,100.00",Ford,Focus,1993,MARTIN
896,O50197197RUS,2,"7,800.00",Ford,Focus,1986,WRIGHT
897,7608EE777RUS,1,"4,000.00",Skoda,Octavia,2013,HILL


* the new dataframe should have all the car numbers that exist in both dataframes - _Full Outer Join_

In [119]:
df_join2 = pd.merge(left=fines, right=owners, on="CarNumber", how="outer")
display(df_join2)

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2.00,"3,200.00",Ford,Focus,"1,989.00",RICHARDSON
1,Y163O8161RUS,2.00,"1,600.00",Ford,Focus,"1,999.00",RICHARDSON
2,E432XX77RUS,1.00,"6,500.00",Toyota,Camry,"1,995.00",ROSS
3,E432XX77RUS,2.00,"13,000.00",Toyota,Camry,"1,992.00",ROSS
4,7184TT36RUS,1.00,"2,100.00",Ford,Focus,"1,984.00",MORGAN
...,...,...,...,...,...,...,...
928,X000Q0000RUS,2.00,"40,000.00",Ford,Passat,"2,004.00",NaN
929,89384CC374RUS,1.00,"1,000.00",Toyota,Camry,"2,025.00",NaN
930,O938284829RUS,NaN,NaN,NaN,NaN,NaN,ANNA
931,P394XXX93hRUS,NaN,NaN,NaN,NaN,NaN,SAVVA


* the new dataframe should have only the car numbers from the fines dataframe - _Left Join_

In [120]:
df_join3 = pd.merge(left=fines, right=owners, on="CarNumber", how="left")
display(df_join3)

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,"2,100.00",Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,"2,000.00",Ford,Focus,2015,BAILEY
4,92918M178RUS,1,"5,700.00",Ford,Focus,2014,LOPEZ
...,...,...,...,...,...,...,...
925,I837GX83923RUS,1,"21,000.00",Toyota,Focus,1999,NaN
926,X777XX7777RUS,2,"3,000.00",Volkswagen,Focus,2000,NaN
927,T834DJ9384734RUS,1,"1,500.00",Toyota,Camry,1998,NaN
928,X000Q0000RUS,2,"40,000.00",Ford,Passat,2004,NaN


* the new dataframe should have only the car numbers from the owners dataframe - _Right Join_

In [121]:
df_join4 = pd.merge(left=fines, right=owners, on="CarNumber", how="right")
display(df_join4)

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2.00,"3,200.00",Ford,Focus,"1,989.00",RICHARDSON
1,Y163O8161RUS,2.00,"1,600.00",Ford,Focus,"1,999.00",RICHARDSON
2,E432XX77RUS,1.00,"6,500.00",Toyota,Camry,"1,995.00",ROSS
3,E432XX77RUS,2.00,"13,000.00",Toyota,Camry,"1,992.00",ROSS
4,7184TT36RUS,1.00,"2,100.00",Ford,Focus,"1,984.00",MORGAN
...,...,...,...,...,...,...,...
897,7608EE777RUS,1.00,"4,000.00",Skoda,Octavia,"2,013.00",HILL
898,7608EE777RUS,2.00,"63,300.00",Skoda,Octavia,"1,987.00",HILL
899,O938284829RUS,NaN,NaN,NaN,NaN,NaN,ANNA
900,P394XXX93hRUS,NaN,NaN,NaN,NaN,NaN,SAVVA


## 8) Create a pivot table from the fines dataframe

In [122]:
display(pd.pivot_table(fines, index=["Make", "Model"], columns="Year", values="Fines", aggfunc={"Fines":sum}))

Year                     1980       1981       1982       1983       1984  \
Make       Model                                                            
Ford       Focus   156,700.00 307,594.59 167,100.00 166,394.59 181,200.00   
           Mondeo         NaN        NaN  46,200.00        NaN        NaN   
           Passat         NaN        NaN        NaN        NaN        NaN   
Skoda      Octavia  21,594.59   1,900.00   8,894.59        NaN   4,900.00   
Toyota     Camry    12,000.00        NaN  21,000.00   2,200.00   1,000.00   
           Corolla        NaN   6,800.00        NaN  12,800.00        NaN   
           Focus          NaN        NaN        NaN        NaN        NaN   
Volkswagen Focus          NaN        NaN        NaN        NaN        NaN   
           Golf     20,800.00   8,594.59   5,000.00     200.00        NaN   
           Jetta          NaN   1,000.00        NaN        NaN        NaN   
           Passat      900.00  49,000.00        NaN   1,100.00   8,594.59   
           Touareg        NaN        NaN        NaN        NaN        NaN   

Year                     1985      1986       1987      1988       1989  ...  \
Make       Model                                                         ...   
Ford       Focus   271,994.59 74,400.00 357,094.59 90,478.36 206,394.59  ...   
           Mondeo         NaN       NaN        NaN       NaN        NaN  ...   
           Passat         NaN       NaN        NaN       NaN        NaN  ...   
Skoda      Octavia  14,694.59       NaN  65,300.00  5,100.00   8,594.59  ...   
Toyota     Camry          NaN 19,800.00        NaN       NaN     800.00  ...   
           Corolla   9,300.00       NaN  54,300.00       NaN   7,800.00  ...   
           Focus          NaN       NaN        NaN       NaN        NaN  ...   
Volkswagen Focus          NaN       NaN        NaN       NaN        NaN  ...   
           Golf    168,000.00       NaN  10,300.00       NaN     300.00  ...   
           Jetta     9,000.00       NaN        NaN 46,000.00  15,400.00  ...   
           Passat         NaN 16,000.00   2,000.00  8,594.59        NaN  ...   
           Touareg        NaN       NaN        NaN       NaN        NaN  ...   

Year                     2010       2011       2012       2013       2014  \
Make       Model                                                            
Ford       Focus   174,200.00 248,294.59 152,589.18 406,289.18 165,683.77   
           Mondeo         NaN        NaN        NaN  41,100.00        NaN   
           Passat         NaN        NaN        NaN        NaN        NaN   
Skoda      Octavia  13,500.00   3,000.00   1,700.00  11,800.00  21,200.00   
Toyota     Camry    22,400.00        NaN   7,500.00        NaN        NaN   
           Corolla   6,000.00  43,600.00        NaN        NaN        NaN   
           Focus          NaN        NaN        NaN        NaN        NaN   
Volkswagen Focus          NaN        NaN        NaN        NaN        NaN   
           Golf           NaN        NaN        NaN        NaN  13,900.00   
           Jetta          NaN        NaN        NaN        NaN        NaN   
           Passat    9,500.00        NaN  27,800.00   1,600.00  12,000.00   
           Touareg        NaN        NaN        NaN        NaN        NaN   

Year                     2015       2016       2017       2018     2025  
Make       Model                                                         
Ford       Focus   253,400.00 163,089.18 103,500.00 166,400.00      NaN  
           Mondeo         NaN        NaN   8,600.00        NaN      NaN  
           Passat         NaN        NaN        NaN        NaN      NaN  
Skoda      Octavia  16,394.59  35,700.00   2,400.00 153,200.00      NaN  
Toyota     Camry          NaN  27,000.00        NaN        NaN 1,000.00  
           Corolla  21,000.00        NaN   2,900.00        NaN      NaN  
           Focus          NaN        NaN        NaN        NaN      NaN  
Volkswagen Focus          NaN        NaN        NaN    

## Save both the fines and owners dataframes to CSV files without an index

In [123]:
fines.to_csv("fines.csv", index=False)
owners.to_csv("owners.csv", index=False)